Introduciamo il middleware orientato ai messaggi (**MOM Message Oriented Middleware**) partendo da un limite di RPC: con una RPC, $A$ invoca un metodo offerto da $B$. Anche se la chiamata è asincrona e $A$ può continuare a fare altro, purché l'operazione vada a buon fine è necessario *che $B$ sia raggiungibile e in grado di riceverla **durante** la chiamata!*

Con **MOM** invece $A$ può depositare un messaggio del tipo "elabora questa foto" e il middleware lo conserva per un certo periodo, per cui $B$ potrà recuperarlo quando lui sarà disponibile. Si parla in questo senso di **comunicazione persistente**, e in termini di decoupling ricordiamo:
- **Decoupling di Sincronizzazione**: *chi invia deve restare bloccato mentre lavora?* MOM lo soddisfa perché $A$, nelle implementazioni asincrone, può continuare a fare altro dopo aver affidato il msg al middleware. *Questo è il decoupling che garantiva anche RPC asincrono*
- **Decoupling temporale**: *mittente e destinatario devono essere attivi nello stesso momento?* MOM lo garantisce grazie alla comunicazione persistente, RPC no
- **Spaziale**: *il mittente deve sapere quale istanza riceverà il messaggio?* Abbiamo visto che in RPC tipicamente il client deve conoscere l'endpoint del server se non c'è un meccanismo di service discovery, mentre in MOM $A$ invia semplicemente alla coda o al topic; è il middleware a gestire la consegna ai destinatari!

Chiaramente, RPC e messaggistica possono convivere nella stessa applicazione, es. un microservizio può usare una RPC per chiedere un dato che gli serve subito e affidare a una coda un lavoro che può essere invece svolto più tardi...

### Message Queue Pattern
Una **coda di messaggi** raccoglie i lavori inviati da dei **producer**. Uno o più **consumer** li prelevano per svolgerli. 

<img src="imgs/7_0.png" width="400">

es. tre worker in grado di ridimensionare una foto. Il producer inserisce in coda un msg per ogni foto --> quando un worker è libero, può prelevare il lavoro. In questo modo il msg relativo a una foto può essere prelevato da worker diversi: in questo senso le code di messaggi si usano per fare **load balancing** tra più istanze di un servizio.  
Altri use case sono *task scheduling*, *logging*, *tracing*...

Questo pattern segue una comunicazione **one-to-one**: ogni msg è prelevato da un **singolo consumer**, anche se possono esestere più consumer che potrebbero prelevarlo.

Tra le operazioni tipiche di una coda di messaggi:
- `put`: il producer mette un messaggio nella coda e può proseguire senza aspettare che il consumer lo elabori
- `get`: il consumer aspetta finché non può recuperare il msg (sincrono). Alcuni sistemi permettono di richiedere un msg che però rispetti certi criteri
- `poll`: il consumer controlla se c'è messaggio. Se la coda è vuota --> riprende subito il proprio lavoro.
- `notify`: il consumer registra un *handler*; quando il msg è disponibile, il middleware lo notifica di modo che il consumer possa reagire.

### Pub/Sub Pattern
Qui cambiamo esigenza: una foto appena caricata interessa a **più componenti** (es. uno deve creare un'anteprima, uno deve aggiornare un indice..). Vogliamo che **entrambi** siano informati dello stesso evento.

Nel pattern pub/sub, il **publisher** pubblica il msg su un **topic** (es. `foto-caricata`) e i **subscriber** esprimono interesse **sottoscrivendo quel topic**. Il middleware consulta le sottoscrizioni e recapita l'evento a tutte quelle che corrispondono: qui si parla di comunicazione **one-to-many**.

Quindi in generale se c'è un lavoro da svolgere e basta che lo faccia un worker --> queue; altrimenti se è accaduto qualcosa e tutti i componenti interessati devono esserne informati --> pub/sub.  

Il componente che si occupa di abbinare messaggi e sottoscrizioni si chiama **event dispatcher**. Questo può considerare anche eventuali **filtri**, ad es. un subscriber può voler ricevere solo un certo tipo di evento dentro un topic. Il dispatcher è un ruolo logico: in un sistema grande è tipicamente necessario che sia realizzato in modo distribuito per evitare SPOF e colli di bottiglia.

Tra le operazioni (API) tipiche: 
- **publish(event)**
- **subscribe(filter, notify_cb, expiry) -> sub_handle**: il subscriber si registra per ricevere eventi di un certo topic, eventualmente indicando un filtro, una callback (notify_cb) e una scadenza.  
Riceve in output un *handle* che può usare per annullare la sottoscrizione tramite **unsubscribe(sub_handle)**.  
Quando un evento corrisponde, il middleware può invocare la callback **notify_cb(sub_handle, event)**.  

### Semantiche e modelli di consegna MOM
Ma come si gestiscono i fallimenti? Immaginiamo un messaggio che contenga l'istruzione "incrementa il contatore della parola `casa`":
- **At-most-once** == **zero oppure una consegna**: il msg può andare perso, ma se arriva il sistema offre comunque la garanzia di consegnarlo al più una volta.
- **At-least-once** == **una o più consegne**, entro le condizioni previste dal sistema. Un meccanismo tipico è far inviare al consumer una conferma (**ACK**), e se il middleware non riceve la conferma può allora riconsegnare il messaggio.. tuttavia il consumer potrebbe aver già incrementato il contatore, da qui nasce il problema dei duplicati
- **Exactly-once** == il msg produce **un solo effetto logico**: qui è da distinguere tra consegna e elaborazione; il msg può anche arrivare due volte, ma il secondo arrivo deve essere gestito affinché vi sia alla fine dei conti un solo incremento. Si deve quindi avere garanzia che anche dopo un guasto ci siano le info necessarie per capire ciò che è stato già elaborato...

Un modo per rendere sicura la ripetizione di un'operazione potrebbe essere quella di assegnare ad ogni msg un ID univoco e ricordare quelli già elaborati (scarto i duplicati), oppure nel caso del contatore invece di dire "incrementa di 1" si può dire "fai il set a tot valore" (es. "fai il set a 5"), trasformando l'operazione in un'**operazione idempotente**.

Altre due semantiche di consegna specifiche di MOM sono:
- **Timeout-based delivery**: garantisce semantica **at-least-once**, senza che però un singolo consumer possa processare più volte il messaggio (at least once perché consumer diversi potrebbero processarlo).  
Il consumer $A$ riceve il msg --> il middleware lo conserva, ma lo rende **invisibile** per un certo periodo di tempo (timeout, es. 60 sec). Durante quel periodo gli altri consumer non possono prelevare il msg.  
Se $A$ termina e invia l'ACK entro il tiemout, il middleware elimina il msg, altrimenti allo scadere del timeout il msg torna disponibile per un altro tentativo
-**Transaction-based delivery**: garantisce che i msg siano cancellati dalla coda solo dopo che sono stati ricevuti e processati dal consumatore.  
L'idea è quindi collegare l'effetto dell'elaborazione e la conferma definitiva del consumo: se la transazione riesce, il *commit* rende **definitivo** il risultato e il sistema può considerare il msg consumato.  
Se fallisce, il *rollback* annulla le modifiche comprese nella transazione e il msg resta recuperabile.  
Questa fgaranzia vale per le operazioni che partecipano alla specifica transazione, ricordiamo questa semantica perché torna tra le configurazioni avanzate di Kafka.

<img src="imgs/7_1.png" width="400">

---

Mentre la semantica di consegna ci dice **quante volte** un msg può arrivare, il **modello di consegna** ci dice **chi prende l'iniziativa**:
- **Push**: il middleware segnala o consegna un msg quando è disponibile --> il consumer può attendere la notifica e reagire quando arriva
- **Pull**: è il consumer a richiedere al middleware se ci sono dati da leggere, ad es. tramite polling. Lo fa quando è pronto ad elaborarne altri.

Vedremo come Kafka usi **pull per i consumer**: il vantaggio è che ciascun consumer può recuperare i dati secondo il proprio ritmo e il middleware risponde alle richieste di lettura. Lo svantaggio è quando la coda è vuota e il consumer interroga troppo spesso il sistema.

### Routing e trasformazione dei msg
Finora abbaimo immaginato una coda facilmente raggiungibile, tuttavia in alcuni sistemi reali le code sono **distribuite**. Immaginiamo un modello dove il mittente deposita il msg nella propria **coda locale** e il destinatario la recupera dalla sua. In mezzo, dei **queue manager** lo inoltrano anche attraverso nodi intermedi.

<img src="imgs/7_2.png" width="400">

In questo senso i queue manager formano una **rete overlay**: ci ricolleghiamo quindi la routing già studiato --> diverse possibilità di instradamento come usare **tabelle di routing** (in una **rete statica**, sono modificate manualmente, in una **dinamica** vengono aggiornate mentre il sistema è in funzione, runtime --> più complesse ma supportano routing scalabile e adattivo), algoritmi di **flooding** etc..

Un altro problema è che il msg può arrivare al componente giusto, ma **in un formato che questo non è in grado di leggere** (es. il producer invia un campo chiamato `costumer_id` mentre un'applicazione già esistente si aspetta `cliente`) --> si può adottare un **broker pattern** dove il **message broker** può applicare regole di conversione e anche scegliere la destinazione in base al contenuto o al tipo di msg.  
Chiaramente anche qui se tutto passasse attraverso un solo broker SPOF e collo di bottiglia --> tipicamente implementato in modo distribuito.

**Coda** e **sub/pub** descrivono il modo in cui vogliamo distribuire i msg. Uno stesso prodotto può offrire entrambi i comportamenti, ad es. **RabbitMQ**, **Kafka**, **Pulsar**, **Redis** e **IBM MQ**. Si distinguono questi strumenti dai **servizi gestiti in cloud**, dove il provider si occupa di gestire buona parte dell'infrastruttura.

Vedremo Amazon SQS come esempio di servizio reale di timeout di visibilità e poi Kafka come esempio di sistema distribuito più in dettaglio.

### Amazon SQS
**Amazon Simple Queue Service** (SQS) è un servizio AWS che gestisce code di messaggi.  
L'idea è quindi che un componente possa lasciare nella coda un lavoro da svolgere e proseguire, e un altro componente preleverà quel lavoro quando avrà capacità per eseguirlo --> disaccoppiamento temporale, possono lavorare in momenti diversi.

AWS si occupa totalmente di gestire la coda, e i messaggi sono distribuiti in **più SQS server all'interno della regione AWS scelta** (es. `eu-west-1` per Dublino).  
Si utilizza un meccanismo di **replicazione** all'interno della stessa regione: i msg sono salvati in più server affinché si garantisca maggiore disponibilità in caso di guasti.  
La consegna in SQS segue **modello pull**: è il consumer che invia una richiesta alla coda per prelevare i messaggi. Ciò si ricollega al fatto che il modello queue è **one-to-one**: possono esserci sì più consumer, ma per ciascun lavoro l'obiettivo è che solo uno di essi lo esegua.

Nell'esempio ogni lettera rappresenta un msg, replicato potenzialmente su più server.

<img src="imgs/7_3.png" width="400">

**Amazon SQS Message Handling**: come anticipato, SQS implementa la semantica di consegna **at-least-once** con **timeout-based delivery**.  
Immaginiamo che la coda contenga un msg $A$ e che un consumer lo prenda a carico. Da quel momento allora il consumer comincia a lavorare mentre SQS rende **temporaneamente invisibile** il msg agli altri consumer. $A$ è però ancora nella coda, e la finestra in cui resta invisibile è detta **visibility timeout**.  
Ci sono due possibili esiti:
- **Il consumer completa il lavoro** --> il consumer chiama l'API `DeleteMessage` per rimuovere il msg dalla coda.
- **Il consumer si blocca, fallisce o impiega oltre il timeout** --> $A$ torna visibile e un altro consumer può prelevarlo per tentare di completare il lavoro. Da qui la semantica **at-least-once**, non è detto che il consumer originale non abbia già completato il lavoro prima di bloccarsi...

Dopo che il msg è stato inserito nella coda, SQS lo conserva per un periodo di tempo detto **retention period** (da non confondere quindi con il visibility timeout: il primo della durata standard di 30 sec, il secondo configurabile fino a 14 giorni).  
Il retention period rappresenta una "scadenza" per il msg: indica per quanto tempo il msg può restare in coda in attesa di un consumer che lo prenda a carico.

**Come si cercano i messaggi in coda e in che ordine arrivano?**  

Il **polling** è il meccanismo standard con cui il consumer attua il modello pull: chiede periodicamente a SQS se ci sono msg disponibili in coda.  
Ricordando però che la coda è distribuita su più server, bisogna distinguere tra due tipi di polling:
- **Short Polling**: SQS consulta un sottinsieme dei server e risponde velocemente. Il problema qui è che potrebbero esserci msg disponibili su server che la richiesta non ha consultato. Il meccanismo per scegliere il sottinsieme dipende dalla policy specifica;
- **Long Polling**: SQS consulta tutti i server e se non c'è msg disponibile può tenere la richiesta aperta per un tempo configurabile fino a 20 sec. Se durante l'attesa arriva un msg allora SQS risponde subito al consumer, altrimenti alla scadenza del timeout risponde con "nessun msg disponibile" (questo tempo serve perché stabilire la connessione a tutti i server richiede tempo, se ho dovuto aprire la connessione con tutti a sto punto aspetto un attimo se trovo vuoto se qualcuno manda qualcosa...)

Per quanto riguarda l'ordine di consegna dei msg, due tipologie principali:
- **Standard**: l'ordine è *best-effort* --> inviando $A, B, C, D$ non è garantito che siano prelevati dal consumer in quell'ordine. La semantica seguita in questo caso è inoltre **at-least-once**, per cui sono possibili duplicati, tuttavia questa filosofia best effort permette *throughput (msg/sec) molto alto*
- **FIFO**: si garantisce l'ordine di consegna al consumer all'interno dello stesso gruppo di messaggi. Per la semantica, con questa configurazione SQS deduplica gli invii ripetuti --> queste garanzie aggiuntive comportano throughput più basso.  
(attendo a non confondere il fatto della deduplicazione: comunque vale timeout based delivery per cui se scade il visibility timeout alcuni messaggi potrebbero essere eseguiti più volte da consumer diversi, il punto della deduplicazione qui riguarda lo stesso consumer)

<img src="imgs/7_4.png" width="400">

**API principali**:
- `CreateQueue, ListQueues e DeleteQueue`: per creare, elencare e cancellare code
- `SendMessage` per inserire un messaggio in coda. Per un contenuto grande, si può **salvare il file su S3 e inviare solo il suo riferimento in coda**
- `ReceiveMessage` per prelevare i msg disponibili. Si può chiedere da 1 a un max di 10 msg per chiamata
- `DeleteMessage` per rimuovere un msg dalla coda dopo che il consumer ha completato il lavoro
- `ChangeMessageVisibility` per modificare il timeout di visibilità di un msg già prelevato. Utile se il consumer si accorge che il lavoro richiederà più tempo del previsto
- `GetQueueAttributes` e `SetQueueAttributes` per ottenere e impostare informazioni sulla coda, es. il numero di msg disponibili, il timeout di visibilità, il retention period etc.

Un concetto su cui vale la pena soffermarsi è il **filtraggio**: con `ReceiveMessage` un consumer non può chiedere messaggi di un certo tipo... può esaminarli e filtrarli **solo dopo averli prelevati**.  
Se ad es. quindi trovo un msg destinato a qualcun altro --> tornerà in coda dopo il visibility timeout... quindi anche una ricezione fatta solo per ispezionare il contenuto incide temporaneamente sulla sua disponibilità.

---

Vediamo un semplice esempio di servizio di elaborazione di immagini implementato sfruttando Amazon SQS:

<img src="imgs/7_5.png" width="450">

Qui **S3 conserva le foto** e **SQS conserva il lavoro da eseguire sulla foto**. Il front-end riceve l'upload dell'utente, mentre il back-end è un insieme di istanze EC2 (VM) che elaborano le immagini:
1. Il front-end riceve la foto e la carica in un bucket S3
2. Il front-end inserisce in SQS un messaggio con il riferimento alla foto (qualsiasi istanza del backend potrà recuperare il file tramite esso)
3. Un'istanza del backend con capacità disponibile preleva il msg --> parte visibility timeout etc..
4. Quando ha finito, cancella il msg da SQS, e se invece si è bloccato prima della cancellazione il lavoro torna disponibile alla scadenza del timeout

In figura vediamo **due gruppi di autoscaling**, uno per il front-end e l'altro per il back-end, ed è la coda che permette proprio di separare le due componenti.  
La coda in questo senso aiuta direttamente l'autoscaling in backend come si vede dalla freccia `Message count`: se il numero di lavori in attesa cresce eccessivamente --> manda il segnale allo scaler che dovrà fare **scale-out** e aggiungere istanze, se invece il numero di msg in coda scende --> manda il segnale per fare **scale-in** e rimuovere istanze.

### Kafka
**Apache Kafka** è una piattaforma distribuita open source per **lo streaming di eventi**.    
Uno stream è un flusso di eventi che arrivano nel tempo, nel lessico della lezione *evento, messaggio e record indicano la stessa cosa*.  
Oggi è uno dei sistemi più usato dato che è scalabile, fault tolerant e permette di gestire un throughput di miliardi di eventi. Scritto in Java e Scala.

Kafka nasce a LinkedIn come sistema **pub/sub** e permette di pubblicare, conservare e leggere flussi di eventi, La piattaforma offre anche strumenti per elaborarli, come Kafka Streams, ma per la lezione ci concentriamo solo **su come gli eventi sono scritti, conservati e consegnati**.

Terminologia in Kafka:
- **Producer**: è l'applicazione che pubblica gli eventi (es. servizio che registra un nuovo ordine)
- **Kafka Topic**: è la categoria in cui vengono pubblicati gli eventi (es. `ordini`)
- **Kafka cluster**: è l'insieme di derver Kafka che gestiscono i topic. Ogni server Kafka è detto **broker** (da non confondere con il message broker di prima che si occupava di routing e trasformazione dei msg!) e può essere distribuito in più data center o regioni cloud. Ogni broker riceve e conserva gli eventi.
- **Consumer**: si sottoscrive a uno o più topic e legge e processa solo gli eventi pubblicati in essi. Ogni topic può avere 0 o più consumer associati.

Essendo un sistema MOM Kafka ne riprende le forme di disaccoppiamento: ogni producer pubblica su un **topic** --> può ignorare l'identità del consumer (decoupling spaziale); gli eventi sono memorizzati --> consumer e producer possono essere attivi in momenti diversi (decoupling temporale); il producer può pubblicare senza attendere che il consumer elabori l'evento (decoupling di sincronizzazione).  
Ricordiamo che essendo pub/sub a differenza di un queue qui il modello è **one-to-many**: un evento pubblicato su un topic può essere letto da più consumer.

#### Logs, partizioni, offset
**Per ogni topic, Kafka mantiene un partitioned log di eventi**.  
Un **log** è in questo caso da vedere come una struttura dati **append-only**, contenente eventi **ordinati in sequenza**. Gli eventi già inseriti sono immutabili (quindi se ad es. un ordine cambia stato, si aggiunge un nuovo evento `ordine spedito` al log, lasciando però il precedente evento `ordine creato`).

Perché "partitioned"? Perché **Kafka divide ciascun topic in un numero configurato di partizioni**, dove ogni partizione ha un proprio log separato. Qui sotto ogni riga è una partizione di uno stesso topic, e a ogni partizione è associato appunto un log separato.

<img src="imgs/7_6.png" width="400">

Dentro ogni partizione, Kafka assegna a ciascun record un **offset**, i.e. un numero che ne identifica univocamente la posizione all'interno del log e che cresce in modo incrementale.  
Nell'esempio si vede come consumer diversi possano partire da offset diversi (uno da 9 e l'altro da 11), e che quando il producer scrive il record successivo allora il broker si occupa di aggiungerlo in fondo (assegnandogli offset 12)

<img src="imgs/7_7.png" width="400">

La divisione in partizioni è quella che permette a Kafka di **scalare**: se implementate fisicamente su broker diversi potranno essere scritte in parallelo da producer diversi e lette in parallelo da consumer diversi!  
In questo senso la partizione è proprio l'**unità di parallelismo** di Kafka: con più partizioni si aumenta il throughput!

<img src="imgs/7_8.png" width="500">

#### Gestione repliche, guasti e coordinamento
Partizionare e replicare risolvono problemi diversi. Partizionare i topic in log distinti, che possono stare poi su broker diversi, permette di aumentare il parallelismo, distribuire il carico e quindi **scalare**. 

La **replicazione** in Kafka invece consiste nel conservare più copie della *stessa partizione* su broker diversi, di modo da recuperare dati e sostituire il leader in caso di guasto --> **improve fault tolerance**

Ogni partizione ha infatti una **replica leader** e può avere una o più repliche follower. *Solo il leader riceve le scritture e serve le letture ai consumer che ne necessitano*, mentre i follower si limitano a copiare i dati dal leader.  
Il **replication factor** conta tutte le copie, leader compreso --> un replication factor di $N$ significa che la partizione può sopportare al più $N-1$ guasti di broker senza perdere dati (assumendo chiaramente che la replica supersite contenga i dati già confermati).

<img src="imgs/7_9.png" width="500">

Ma perché un solo leader per partizione? Il motivo è che così è più semplice garantire la consistenza dei dati: se ad es. due producer scrivessero contemporaneamente $A$ e $B$ su due repliche leader diverse della stessa partizione --> se non è implementato un meccanismo di coordinamento una potrebbe scrivere $A,B$ e l'altra $B,A$. Con un unico leader la necessità di correggere questo problema di consistenza non si pone.

Ma come ricevono gli aggiornamenti i follower? Consideriamo tre possibilità, una sola implementata in Kafka:
1. Il leader invia ogni nuovo record a tutti i follower. Ciò comporta lavoro di invio ad ogni scrittura... intrattabile se il throughput è alto
2. Il leader accumula record in batch periodici e invia il blocco. Riduce il numero di invii, ma deve ancora gestire come contattare tutti i follower
3. **I follower fanno pull al leader per i nuovi record**: è la scelta di Kafka, il follower di turno chiede i dati a partire dal punto in cui è arrivato e li aggiunge al proprio log.  
Ogni follower può essere *in-sync* se è a pari passo con il leader, *out-of-sync* se è indietro in termini di indice di offset. Se un follower è out-of-sync per troppo tempo Kafka può decidere di escluderlo dalla lista dei follower.

Ma avendo un solo leader... può diventare collo di bottiglia? Per una singola partizione sì, dal momento che le scritture passano tutte da lui.  
A livello di topic però ricordiamo che Kafka ha più partizioni e quindi anche leadership potenzialmente distribuite in broker diversi --> se si guasta un broker, le partizioni follower su broker diversi possono diventare leader secondo un protocollo distribuito di **leader-election** (di questa gestione se ne occupa una componente separata, *KRaft*, con protocollo di consenso Raft) e continuare a servire le richieste. 

In figura si vede come l'esplosione atomica del Broker 1 fa si che il vecchio follower di Partition 0, Topic B diventi leader e si passi a una successiva nuova replica in un altro broker (dato che la replication factor in questo esempio è fissa a 2).

<img src="imgs/7_10.png" width="500">

**Come fa il producer a scegliere dove scrivere?**

Il producer sceglie il topic e determina la partizione dove scrivere il record. Una volta ottenuti i metadati, invia il record **direttamente** al broker leader di quella partizione (i.e. il broker che ha la replica leader), senza quindi seguire alcuna strategia di routing (i metadati servono proprio per ottenere le informazioni necessarie per raggiungere direttamente il broker d'interesse).

Tre politiche per fare ciò:
1. **Per chiave**: ogni evento (es. `order ID`) ha una **partition key** --> il producer, facendo l'hash della chiave, ottiene la partizione dove scrivere. In questo modo a parità di configurazione si garantisce che gli stessi eventi dello stesso utente finiscono sempre nella stessa partizione --> hanno ordine comune nel log
2. **Senza chiave**: si usa in questo caso **round-robin**, alternando le partizioni. Nelle implementazioni dei producer Java più recenti **politica sticky** per cui round robin però a batch (senza quindi che per ogni singolo evento si cambi partizione --> invio più efficiente)
3. **Custom partitioning**: l'applicazione si occupa di fornire una propria logica per determinare la partizione di destinazione

Chiaramente più producer possono inviare eventi **contemporaneamente alla stessa partizione**: il leader li aggiunge al log nell'ordine in cui vengono accettati ed assegna a loro i relaitiv offset.

**Come il consumer legge e riparte dopo un guasto**

Come in SQS, è il consumer a fare **pull** dei record dal broker leader della partizione. In Kafka la logica è "dammi i record di questa partizione a partire da questo offset".  
Due applicazioni che consumano indipendentemente possono leggere la stessa partizione a velocità diverse (es. la fatturazione può già essere all'offset 15 mentre l'applicazione di analisi al 10).

Un modello push sarebbe impensabile: il broker dovrebbe decidere quando inviare i dati a ciascun consumer e a quale ritmo, nonché sapere a che offset ciascun consumer è arrivato.  
Con il pull invece ciascun consumer richiede dati quando è pronto e può adeguare il ritmo di lettura alla propria capacità.  
Il costo in ciò è che ogni consumer deve interrogare Kafka per sapere se ci sono nuovi eventi --> possibilità di **busy waiting** (il consumer fa richieste continue anche se non ci sono dati disponibili), per mitigarlo Kafka permette alle richieste di lettura di specificare un timeout prima di "arrendersi".

Qui un problema: ogni consumer deve ricordarsi dell'offset a cui è arrivato nel log. Se lo salvasse nella propria memoria (volatile) --> in caso di crash perderebbe il riferimento. Per tale ragione ogni consumer può **fare commit dell'offset** su un topic speciale interno di Kafka: `__consumer_offsets` (anch'esso partizionato e replicato per fault tolerance).  
Il client Java abilita di default *l'auto commit periodico* dell'offset, e convenzionalmente si fa commit del prossimo offset da leggere.  
Chiaro che se un client ha elaborato fino all'offset 15 ma va in crash prima di fare commit 16 si rischia di rielaborare il record 15 al riavvio...

**Retention period e segmenti**

Avevamo visto in SQS come a lavoro completato il msg venisse cancellato dal consumer via `DeleteMessage`. In Kafka è diverso ovviamente, dal momento che si tratta di un'implementazione **pub/sub** e non queue è **one-to-many** --> dopo una lettura di un record *non è detto che tutti i consumer abbiano finito di leggerlo, quindi non si può cancellare subito!*

La retention in Kafka quindi determina **quanto tempo o entro quale limite di spazio Kafka conserva i dati nel log, così che più applicazioni possano leggerli in momenti diversi**. 

Su disco ogni partizione è suddivisa in **segmenti di log**, i.e. file che contengono più record consecutivi:
- con una politica di cancellazione basata sul **tempo** --> Kafka rimuove i segmenti più vecchi quando scadono
- con una politica basata sulla **dimensione** --> rimuove i segmenti più vecchi quando il log della partizione supera il limite configurato. 

Chiaramente quindi è una policy da configurare in base all'applicazione: attenzione al fatto che consumer lenti possono perdere la possibilità di recuperare record se Kafka li ha già cancellati!

Per cercare di ridurre spazio una possibilità è la **log compaction**: immaginiamo un sensore che invia temperature successive usando sempre come chiave l'ID del sensore. Se ci interessa solo ricostruire le temperature più recenti per ogni sensore --> Kafka permette di configurare la **log compaction** per cui Kafka può rimuovere solo i record più vecchi associati alla stessa chiave. In questo modo anche se ho un offset molto vecchio ma associato a una chiave che non è più stata aggiornata quella resta disponibile (proprio perché voglio le ultime temperature per ogni sensore!).  
Compaction quindi utile **quando ci interessa lo stato più recente per chiave**.

Un altro elemento importante è il **flush**.  
Quando il broker aggiunge dati al log, il SO può tenerli inizialmente nella **page cache** in RAM. Il **flush** si occupa di portare quei dati su disco --> si deve trovare un equilibrio tra performance (trasferire continuamente su disco è costoso, sarebbe preferibile raggruppare più scritture) e sicurezza (se il broker va in crash prima del flush, i dati in RAM vanno persi, sarebbe preferibile fare flush più spesso)... 